# Train the YOLO detection models (crop + RVIP) — baseline

Drop-in replacement for the two nnUNet **detection** stages, keeping the rest of the
pipeline (upstream preprocessing + LV segmentation) unchanged so it is a fair comparison:

- **CROP model** (Dataset110) — step-1 whole-heart ROI in the full image.
- **RVIP model** (Dataset105) — the 2 RV insertion points, from the IPs-only mask (`IP1 = 1` anterior, `IP2 = 2` inferior; no LV).

The LV segmentation stays nnUNet (Dataset100, trained separately). This notebook only
**trains** the two detectors and sanity-checks the labels; wiring YOLO into
`inference.ipynb` is a separate step.

**Baseline = first contrast only.** YOLO reads the *same* `imagesTr` NIfTIs nnUNet trains
on (the output of the existing preprocessing) and uses the first contrast `_0000`
(`CHANNELS = (0, 0, 0)`, replicated to R/G/B — YOLO needs 8-bit RGB). The multi-contrast
extension `(0, 1, 2)` stays available for a later study but is **not** the baseline.

**Test data is off-limits:** only `imagesTr`/`labelsTr` are read here. `imagesTs`/`labelsTs`
are never touched.

**No manual labelling** — labels are derived from the existing nnUNet ground truth by `yolo_pipeline.py`.

In [ ]:
import os, glob, json, sys
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

# yolo_pipeline.py lives in the repo root next to this notebook
sys.path.append(os.getcwd())
import yolo_pipeline as yp

# YOLO backend (ultralytics is only needed for training/inference, not for the export)
try:
    import ultralytics
    print('ultralytics', ultralytics.__version__)
except ImportError:
    print('ultralytics not installed ->  pip install ultralytics')

## 1. Config
Point these at your ground-truth nnUNet datasets and pick hyperparameters.
Everything below is derived from these.

In [ ]:
# =====================================================================
#  RVIP model  ->  train on Dataset105 (combined LV + insertion points)
# =====================================================================
# imagesTr holds the contrasts (*_0000 first). Baseline uses ONLY _0000.
# CONFIRM label ids against dataset.json (see RVIP_LABEL_IDS below).
# ---- pick datasets by NAME; everything else derives from these ----
# Switch RVIP_DATASET / CROP_DATASET to train on different data (e.g. Smart Health + extra data).
# Run folders are tagged with the dataset (rvip_D105, crop_D110, ...) so different datasets don't
# overwrite each other's weights and are easy to compare.
NNUNET_RAW  = '/home/sastocke/nnUNet/nnUNet_raw'
NNUNET_PREP = '/home/sastocke/nnUNet/nnUNet_preprocessed'
RVIP_DATASET = 'Dataset105_HannumSmartHealthDataIPs'     # <-- RVIP training data
CROP_DATASET = 'Dataset110_HannumSmartHealthDataCrop'    # <-- CROP training data
RVIP_TAG = 'D' + RVIP_DATASET.split('_')[0].replace('Dataset', '')   # e.g. 'D105'
CROP_TAG = 'D' + CROP_DATASET.split('_')[0].replace('Dataset', '')

RVIP_IMAGES = f'{NNUNET_RAW}/{RVIP_DATASET}/imagesTr'
RVIP_LABELS = f'{NNUNET_RAW}/{RVIP_DATASET}/labelsTr'
RVIP_SPLITS = f'{NNUNET_PREP}/{RVIP_DATASET}/splits_final.json'   # train/val; or None
# Insertion-point label ids (Dataset105 IPs-only convention: IP1=1 anterior, IP2=2 inferior)
RVIP_ANTERIOR_LABEL = 1
RVIP_INFERIOR_LABEL = 2
# Single-class RVIP: detect both points as one class, then assign anterior/inferior by geometry
# (anterior = higher). fliplr aug scrambles a 2-class cue, so single-class is recommended.
RVIP_SINGLE_CLASS = True
RVIP_FLIP_ASSIGN  = False   # set True if a GT check shows anterior/inferior are swapped
CROP_IMAGES = f'{NNUNET_RAW}/{CROP_DATASET}/imagesTr'
CROP_LABELS = f'{NNUNET_RAW}/{CROP_DATASET}/labelsTr'
CROP_SPLITS = f'{NNUNET_PREP}/{CROP_DATASET}/splits_final.json'   # or None
OUT_ROOT = '/home/sastocke/nnUNet/yolo_datasets'   # YOLO-format data + runs go here
FOLD     = 2                                        # which split fold to mirror

# ---------- channel selection (nnUNet contrast index -> YOLO R/G/B) ----------
# Baseline: first contrast only, replicated to R/G/B (YOLO needs 8-bit RGB).
#   (0, 0, 0) -> use _0000 only  [BASELINE for both models]
#   (0, 1, 2) -> three distinct contrasts  [multi-contrast extension; RVIP only]
CHANNELS      = (0, 0, 0)    # RVIP (Dataset105): first-contrast baseline
CROP_CHANNELS = (0, 0, 0)    # CROP (Dataset110): single contrast

# ---------- hyperparameters ----------
RVIP_IMGSZ   = 256          # YOLO letterboxes to this (multiple of 32)
CROP_IMGSZ   = 256
RVIP_BOX_PX  = 25           # px box around each insertion point (bigger -> easier to detect/localize)
EPOCHS       = 100
YOLO_WEIGHTS = 'yolov8n.pt'  # nano is plenty for single-structure / 2-point detection

assert len(CHANNELS) == 3 and len(CROP_CHANNELS) == 3, 'CHANNELS must have 3 entries (R, G, B).'
tag = lambda ch: '(multi-contrast extension)' if len(set(ch)) > 1 else '(first-contrast baseline)'
print('RVIP CHANNELS =', CHANNELS, tag(CHANNELS))
print('CROP CHANNELS =', CROP_CHANNELS, tag(CROP_CHANNELS))

## 2. Helper functions
`build_train_val` pairs `imagesTr` cases with `labelsTr` and assigns train/val from the
nnUNet split (test set is never touched); `show_yolo` overlays the exported boxes so you
can eyeball that the labels are correct.

In [ ]:
def load_split_map(splits_json, fold=0):
    """Return {case_id: 'train'|'val'} from an nnUNet splits_final.json."""
    with open(splits_json) as f:
        splits = json.load(f)
    m = {c: 'train' for c in splits[fold]['train']}
    m.update({c: 'val' for c in splits[fold]['val']})
    return m

def build_samples(images_dir, labels_dir, split_map=None, channel='_0000', default_split='train'):
    """Pair imagesXx channel-0 files with their labels; attach a split label.

    Only the *_0000 file is listed here; yolo_pipeline resolves the other
    contrasts (_0001/_0002/...) from CHANNELS at export/inference time.
    """
    samples = []
    for img in sorted(glob.glob(os.path.join(images_dir, f'*{channel}.nii.gz'))):
        case = os.path.basename(img).replace(f'{channel}.nii.gz', '')
        lbl  = os.path.join(labels_dir, case + '.nii.gz')
        if not os.path.exists(lbl):
            continue
        split = split_map.get(case, default_split) if split_map else default_split
        samples.append({'image_path': img, 'mask_path': lbl, 'name': case, 'split': split})
    return samples

def build_train_val(images_tr, labels_tr, splits_json=None, fold=0):
    """Pair imagesTr with labelsTr and split train/val. Test set is never read."""
    split_map = load_split_map(splits_json, fold) if splits_json else None
    return build_samples(images_tr, labels_tr, split_map)

def show_yolo(img_dir, lbl_dir, n=3, title=''):
    """Overlay exported YOLO boxes on their images to verify labels look right."""
    from PIL import Image
    for p in sorted(glob.glob(os.path.join(img_dir, '*.png')))[:n]:
        im = np.array(Image.open(p)); H, W = im.shape[:2]
        txt = os.path.join(lbl_dir, os.path.basename(p).replace('.png', '.txt'))
        plt.figure(figsize=(4, 4)); plt.imshow(im)   # RGB PNG (first contrast, replicated)
        if os.path.exists(txt):
            for line in open(txt):
                cls, cx, cy, w, h = map(float, line.split())
                plt.gca().add_patch(plt.Rectangle(((cx-w/2)*W, (cy-h/2)*H), w*W, h*H,
                                                  fill=False, edgecolor='r', lw=2))
        plt.title(f'{title} {os.path.basename(p)}'); plt.axis('off'); plt.show()

## 3. RVIP model (Dataset105) — build, export, sanity-check, train

Insertion points come from the IPs-only mask (`IP1 = 1` anterior, `IP2 = 2` inferior).
Each becomes a `RVIP_BOX_PX` box; a slice with no annotated point is skipped. Images are
the first contrast (`_0000`) as RGB PNGs.

**Single-class formulation** (`RVIP_SINGLE_CLASS = True`): both points are exported under
one `insertion_point` class. The 2-class version failed (mAP≈0.06) because anterior vs
inferior can't be told apart from a tiny local patch. Here YOLO just *finds* the two
points; `detect_rvips(single_class=True)` then labels them anterior/inferior by height
(anterior = higher). Check the `val_batch*_pred.jpg` after training; if anterior/inferior
come out swapped, set `RVIP_FLIP_ASSIGN = True`.

In [ ]:
rvip_out = os.path.join(OUT_ROOT, f'rvip_{RVIP_TAG}')
rvip_samples = build_train_val(RVIP_IMAGES, RVIP_LABELS, RVIP_SPLITS, FOLD)
print('rvip samples:', len(rvip_samples), '| split:', Counter(s['split'] for s in rvip_samples))

# nifti (_0000) -> RGB png + insertion-point boxes per slice.
# single_class=True -> both points exported as one 'insertion_point' class.
counts_r = yp.export_dataset(rvip_samples, rvip_out, kind='rvip',
                             box_px=RVIP_BOX_PX, channels=CHANNELS,
                             anterior_label=RVIP_ANTERIOR_LABEL,
                             inferior_label=RVIP_INFERIOR_LABEL,
                             single_class=RVIP_SINGLE_CLASS)
rvip_yaml = yp.write_data_yaml(rvip_out, kind='rvip', single_class=RVIP_SINGLE_CLASS)
print('wrote', rvip_yaml, counts_r)

In [ ]:
# sanity-check: the two red boxes should sit on the insertion points
show_yolo(os.path.join(rvip_out, 'images/train'), os.path.join(rvip_out, 'labels/train'), title='RVIP')

In [ ]:
rvip_model = yp.train_yolo(rvip_yaml, model=YOLO_WEIGHTS, epochs=EPOCHS, imgsz=RVIP_IMGSZ,
                           project=os.path.join(OUT_ROOT, 'runs'), name=f'rvip_{RVIP_TAG}')
print('best weights ->', os.path.join(OUT_ROOT, 'runs', f'rvip_{RVIP_TAG}', 'weights', 'best.pt'))

## 4. CROP model (Dataset110) — build, export, sanity-check, train

Step-1 whole-heart ROI, the drop-in replacement for the Stage-1 crop nnUNet. Baseline uses
the first contrast only, so `CROP_CHANNELS = (0, 0, 0)` (`_0000` replicated into R/G/B). The
GT box is the squared whole-heart mask (`Heart = 1`).

In [ ]:
crop_out = os.path.join(OUT_ROOT, f'crop_{CROP_TAG}')
crop_samples = build_train_val(CROP_IMAGES, CROP_LABELS, CROP_SPLITS, FOLD)
print('crop samples:', len(crop_samples), '| split:', Counter(s['split'] for s in crop_samples))

# single-contrast nifti -> grayscale-as-RGB png + one squared whole-heart box per slice
counts_c = yp.export_dataset(crop_samples, crop_out, kind='crop', channels=CROP_CHANNELS)
crop_yaml = yp.write_data_yaml(crop_out, kind='crop')
print('wrote', crop_yaml, counts_c)

In [ ]:
show_yolo(os.path.join(crop_out, 'images/train'), os.path.join(crop_out, 'labels/train'), title='CROP')

In [ ]:
crop_model = yp.train_yolo(crop_yaml, model=YOLO_WEIGHTS, epochs=EPOCHS, imgsz=CROP_IMGSZ,
                           project=os.path.join(OUT_ROOT, 'runs'), name=f'crop_{CROP_TAG}')
print('best weights ->', os.path.join(OUT_ROOT, 'runs', f'crop_{CROP_TAG}', 'weights', 'best.pt'))

## 5. Quick smoke test (optional)
Load the trained weights and run the notebook-facing adapters on one case each, to confirm
they emit the artifacts the pipeline expects: RVIP -> two points (or `None`); CROP -> a
squared box. The `*_from_path` helpers load the SAME `CHANNELS` you trained with.

In [ ]:
# RVIP: first-contrast image -> two points (anterior/inferior assigned by geometry)
rvip_m = yp.load_model(os.path.join(OUT_ROOT, 'runs', 'rvip', 'weights', 'best.pt'))
pts = yp.detect_rvips_from_path(rvip_m, rvip_samples[0]['image_path'], channels=CHANNELS,
                                single_class=RVIP_SINGLE_CLASS, flip_assignment=RVIP_FLIP_ASSIGN)
print('detected RVIPs:', pts)

# CROP: single-contrast full image -> square box -> binary crop mask (drop-in for Stage 1)
crop_m = yp.load_model(os.path.join(OUT_ROOT, 'runs', 'crop', 'weights', 'best.pt'))
box = yp.detect_crop_box_from_path(crop_m, crop_samples[0]['image_path'], channels=CROP_CHANNELS)
print('detected crop box (x_min,x_max,y_min,y_max,scale):', box)

## Next step (separate task)
Wiring the two models into `inference.ipynb`:
- replace **cell 1** (crop nnUNet) with `detect_crop_box_from_path(..., channels=CROP_CHANNELS)` -> `crop_box_to_mask`;
- replace **cell 5** (IP nnUNet) with `detect_rvips_from_path(..., channels=CHANNELS)` -> `rvips_to_mask` (labels 2/3);
- keep cells 2/3/6/7 (LV = Dataset100), run analysis in original spacing.

Use the **same `CHANNELS` at inference as at training** — baseline is `(0,0,0)` for both
(first contrast). Low-confidence RVIP detections return `None` (a clean miss, not a wild
outlier) — decide there whether to skip or report a failed point.

## 7. Multi-contrast RVIP model (Dataset105, contrasts 0/1/2)

Additional RVIP model that feeds **three contrasts** as R/G/B (`RVIP_MC_CHANNELS=(0,1,2)`)
instead of replicating `_0000`. The RV insertion points sit where the RV wall meets the LV,
which a single contrast may barely show — extra contrasts give the detector more to lock
onto. Everything above (single-contrast crop + RVIP) is untouched; this trains a *separate*
model in `runs/rvip_mc` so you can chain whichever you prefer.

In [ ]:
RVIP_MC_CHANNELS = (0, 1, 2)                    # three distinct contrasts -> R/G/B
rvip_mc_out = os.path.join(OUT_ROOT, f'rvip_mc_{RVIP_TAG}')

rvip_mc_samples = build_train_val(RVIP_IMAGES, RVIP_LABELS, RVIP_SPLITS, FOLD)
print('rvip_mc samples:', len(rvip_mc_samples), '| split:', Counter(s['split'] for s in rvip_mc_samples))

counts_mc = yp.export_dataset(rvip_mc_samples, rvip_mc_out, kind='rvip',
                              box_px=RVIP_BOX_PX, channels=RVIP_MC_CHANNELS,
                              anterior_label=RVIP_ANTERIOR_LABEL,
                              inferior_label=RVIP_INFERIOR_LABEL,
                              single_class=RVIP_SINGLE_CLASS)
rvip_mc_yaml = yp.write_data_yaml(rvip_mc_out, kind='rvip', single_class=RVIP_SINGLE_CLASS)
print('wrote', rvip_mc_yaml, counts_mc)

In [ ]:
# sanity-check: PNGs are now real 3-contrast RGB (not grayscale); boxes on the IPs
show_yolo(os.path.join(rvip_mc_out, 'images/train'), os.path.join(rvip_mc_out, 'labels/train'), title='RVIP-MC')

In [ ]:
rvip_mc_model = yp.train_yolo(rvip_mc_yaml, model=YOLO_WEIGHTS, epochs=EPOCHS, imgsz=RVIP_IMGSZ,
                              project=os.path.join(OUT_ROOT, 'runs'), name=f'rvip_mc_{RVIP_TAG}')
print('best weights ->', os.path.join(OUT_ROOT, 'runs', f'rvip_mc_{RVIP_TAG}', 'weights', 'best.pt'))

## 8. Point-distance evaluation on the TEST set

The metric that matters for the pipeline is **how far the detected point is from the true
insertion point** — not mAP. Here we run a trained RVIP model on the **held-out test set**
(`imagesTs`/`labelsTs`, used for evaluation only) and report the Euclidean distance between
each predicted box-center and the GT centroid, in pixels of the 256 crop.

`euclidean` returns `None` when a point wasn't detected, so misses are counted separately
rather than faked as a distance. The summary also reports what the distance *would* be with
the anterior/inferior assignment swapped — if that's consistently smaller, set
`RVIP_FLIP_ASSIGN=True` (a post-hoc flip, no retrain).

In [ ]:
# Build TEST samples (evaluation ONLY -- training never reads these).
def build_test_samples(images_ts, labels_ts, channel='_0000'):
    samples = []
    for img in sorted(glob.glob(os.path.join(images_ts, f'*{channel}.nii.gz'))):
        case = os.path.basename(img).replace(f'{channel}.nii.gz', '')
        lbl  = os.path.join(labels_ts, case + '.nii.gz')
        if os.path.exists(lbl):
            samples.append({'image_path': img, 'mask_path': lbl, 'name': case})
    return samples

RVIP_IMAGES_TS = RVIP_IMAGES.replace('imagesTr', 'imagesTs')
RVIP_LABELS_TS = RVIP_LABELS.replace('labelsTr', 'labelsTs')
test_samples = build_test_samples(RVIP_IMAGES_TS, RVIP_LABELS_TS)
print('test samples:', len(test_samples))

In [ ]:
def evaluate_points(model, samples, channels, single_class=True, flip=False, conf=0.25):
    """Per-case predicted-vs-GT distances for anterior/inferior insertion points."""
    rows = []
    for s in samples:
        gt = yp.insertion_points_from_mask(s['mask_path'],
                                           anterior_label=RVIP_ANTERIOR_LABEL,
                                           inferior_label=RVIP_INFERIOR_LABEL)
        pred = yp.detect_rvips_from_path(model, s['image_path'], channels=channels,
                                         single_class=single_class, flip_assignment=flip,
                                         conf_thresh=conf)
        rows.append({
            'name': s['name'],
            'd_ant': yp.euclidean(pred['anterior'], gt['anterior']),
            'd_inf': yp.euclidean(pred['inferior'], gt['inferior']),
            # distances if the anterior/inferior assignment were flipped:
            'd_ant_sw': yp.euclidean(pred['inferior'], gt['anterior']),
            'd_inf_sw': yp.euclidean(pred['anterior'], gt['inferior']),
        })
    return rows

def summarize(rows, label=''):
    def stats(vals):
        v = [x for x in vals if x is not None]
        miss = sum(1 for x in vals if x is None)
        return (np.median(v) if v else float('nan'),
                np.mean(v) if v else float('nan'), miss)
    ma = stats([r['d_ant'] for r in rows]); mi = stats([r['d_inf'] for r in rows])
    # mean of the swapped assignment, to hint whether RVIP_FLIP_ASSIGN should be flipped
    norm = np.nanmean([x for r in rows for x in (r['d_ant'], r['d_inf']) if x is not None])
    swap = np.nanmean([x for r in rows for x in (r['d_ant_sw'], r['d_inf_sw']) if x is not None])
    print(f'== {label} ==  (n={len(rows)}, distances in px of the 256 crop)')
    print(f'  anterior: median {ma[0]:.1f}  mean {ma[1]:.1f}  misses {ma[2]}/{len(rows)}')
    print(f'  inferior: median {mi[0]:.1f}  mean {mi[1]:.1f}  misses {mi[2]}/{len(rows)}')
    print(f'  mean dist {norm:.1f}  |  if assignment swapped: {swap:.1f}'
          + ('   <- swap is better; set RVIP_FLIP_ASSIGN=True' if swap + 1e-6 < norm else ''))

In [ ]:
# Compare single-contrast vs multi-contrast RVIP on the test set.
sc_model = yp.load_model(os.path.join(OUT_ROOT, 'runs', 'rvip', 'weights', 'best.pt'))
mc_model = yp.load_model(os.path.join(OUT_ROOT, 'runs', 'rvip_mc', 'weights', 'best.pt'))

sc_rows = evaluate_points(sc_model, test_samples, channels=CHANNELS,
                          single_class=RVIP_SINGLE_CLASS, flip=RVIP_FLIP_ASSIGN)
mc_rows = evaluate_points(mc_model, test_samples, channels=RVIP_MC_CHANNELS,
                          single_class=RVIP_SINGLE_CLASS, flip=RVIP_FLIP_ASSIGN)
summarize(sc_rows, 'single-contrast RVIP (_0000)')
summarize(mc_rows, 'multi-contrast RVIP (0,1,2)')

## 9. Train BOTH RVIP variants for a dataset — by ID

Give a **dataset ID**; this resolves `nnUNet_raw/Dataset<ID>_*`, reads its `dataset.json`
(channels + labels), and trains **two** insertion-point models from the same data:

- **`rvip_1c_D<ID>`** — single contrast `(0,0,0)`
- **`rvip_mc_D<ID>`** — multi-contrast `RVIP_MC_CHANNELS_CFG` (default `(0,1,2)`)

Note: YOLOv8 is a **3-channel** network (pretrained RGB), so "all 4 contrasts" maps to a
3-channel multi-contrast model — `_0003` (FA) is left out. Change `RVIP_MC_CHANNELS_CFG` to
pick which three contrasts to feed. To train a different dataset, just change `RVIP_DATASET_ID`.

In [ ]:
import json

# ==== give it a dataset ID -> trains a 1-contrast AND a multi-contrast RVIP model ====
RVIP_DATASET_ID      = 105            # <-- switch to train IP models on a different dataset
RVIP_MC_CHANNELS_CFG = (0, 1, 2)      # multi-contrast mapping (YOLO is 3-channel; FA=_0003 excluded)

def resolve_dataset(dataset_id):
    hits = sorted(glob.glob(f'{NNUNET_RAW}/Dataset{dataset_id:03d}_*'))
    assert hits, f'no Dataset{dataset_id:03d}_* under {NNUNET_RAW}'
    ddir = hits[0]
    with open(os.path.join(ddir, 'dataset.json')) as f:
        dj = json.load(f)
    print(f'Dataset{dataset_id:03d}: {os.path.basename(ddir)}')
    print('  channel_names:', dj.get('channel_names'))
    print('  labels       :', dj.get('labels'))
    return ddir, dj

def train_rvip_variants(dataset_id, box_px=RVIP_BOX_PX, epochs=EPOCHS, imgsz=RVIP_IMGSZ):
    ddir, dj = resolve_dataset(dataset_id)
    images = f'{ddir}/imagesTr'
    labels = f'{ddir}/labelsTr'
    splits = f'{NNUNET_PREP}/{os.path.basename(ddir)}/splits_final.json'
    splits = splits if os.path.exists(splits) else None      # None -> everything in train
    tag = f'D{dataset_id}'
    trained = {}
    for variant, channels in [('1c', (0, 0, 0)), ('mc', RVIP_MC_CHANNELS_CFG)]:
        out = os.path.join(OUT_ROOT, f'rvip_{variant}_{tag}')
        samples = build_train_val(images, labels, splits, FOLD)
        print(f'\n=== RVIP {variant} {channels} on {tag}: {len(samples)} samples ===')
        yp.export_dataset(samples, out, kind='rvip', box_px=box_px, channels=channels,
                          anterior_label=RVIP_ANTERIOR_LABEL, inferior_label=RVIP_INFERIOR_LABEL,
                          single_class=RVIP_SINGLE_CLASS)
        data_yaml = yp.write_data_yaml(out, kind='rvip', single_class=RVIP_SINGLE_CLASS)
        yp.train_yolo(data_yaml, model=YOLO_WEIGHTS, epochs=epochs, imgsz=imgsz,
                      project=os.path.join(OUT_ROOT, 'runs'), name=f'rvip_{variant}_{tag}')
        best = os.path.join(OUT_ROOT, 'runs', f'rvip_{variant}_{tag}', 'weights', 'best.pt')
        print('best weights ->', best)
        trained[variant] = best
    return trained

rvip_models = train_rvip_variants(RVIP_DATASET_ID)
print('\ntrained RVIP models for this dataset:')
for v, w in rvip_models.items():
    print(f'  {v}: {w}')


## 10. Contrast ablation — which channels help RVIP detection?

Trains + evaluates every channel config for the dataset in `ABLATION_DATASET_ID`, judged by the
**§8 point-distance metric** (px from GT insertion point) on the held-out test set — *not* mAP.
Each config -> a tagged model (`rvip_D<ID>_<a-b-c>`) + a row per (config, case). Results go to a
CSV and a ranked violin so you can see which single contrast and which triple actually win.

Prior (mine): singles `avg >= FA > E1 > MD`; best triple likely `(0,2,3)` avg+E1+FA (drop MD) —
and single `avg` may rival any triple, given `(0,0,0)` already beat `(0,1,2)` earlier. Let the data decide.

*(Run after §8 so `build_test_samples` / `evaluate_points` are defined. ~1 min per model.)*

In [ ]:
import pandas as pd

# ====== contrast ablation ======
ABLATION_DATASET_ID = RVIP_DATASET_ID          # dataset to ablate (reuses §9's ID)
ABLATION = [
    (0, 0, 0), (1, 1, 1), (2, 2, 2), (3, 3, 3),        # each single contrast (replicated to RGB)
    (0, 1, 2), (0, 1, 3), (0, 2, 3), (1, 2, 3),        # every triple of the 4 contrasts
]
ABLATION_CONF = 0.01                            # low conf: single-class takes top-2 (avoid all-misses)

def _tag(ch): return '-'.join(map(str, ch))     # (0,2,3) -> '0-2-3'

# resolve the dataset + build train/test once
_addir, _adj = resolve_dataset(ABLATION_DATASET_ID)
_ad_images, _ad_labels = f'{_addir}/imagesTr', f'{_addir}/labelsTr'
_ad_splits = f'{NNUNET_PREP}/{os.path.basename(_addir)}/splits_final.json'
_ad_splits = _ad_splits if os.path.exists(_ad_splits) else None
_ad_test = build_test_samples(f'{_addir}/imagesTs', f'{_addir}/labelsTs')
print(f'ablation on Dataset{ABLATION_DATASET_ID}: {len(_ad_test)} test cases, {len(ABLATION)} configs')

rows = []
for ch in ABLATION:
    tag = f'D{ABLATION_DATASET_ID}_{_tag(ch)}'
    out = os.path.join(OUT_ROOT, f'rvip_{tag}')
    print(f'\n===== ablation {ch} -> rvip_{tag} =====')
    samples = build_train_val(_ad_images, _ad_labels, _ad_splits, FOLD)
    yp.export_dataset(samples, out, kind='rvip', box_px=RVIP_BOX_PX, channels=ch,
                      anterior_label=RVIP_ANTERIOR_LABEL, inferior_label=RVIP_INFERIOR_LABEL,
                      single_class=RVIP_SINGLE_CLASS)
    data_yaml = yp.write_data_yaml(out, kind='rvip', single_class=RVIP_SINGLE_CLASS)
    yp.train_yolo(data_yaml, model=YOLO_WEIGHTS, epochs=EPOCHS, imgsz=RVIP_IMGSZ,
                  project=os.path.join(OUT_ROOT, 'runs'), name=f'rvip_{tag}')
    model = yp.load_model(os.path.join(OUT_ROOT, 'runs', f'rvip_{tag}', 'weights', 'best.pt'))
    for r in evaluate_points(model, _ad_test, channels=ch, single_class=RVIP_SINGLE_CLASS,
                             flip=RVIP_FLIP_ASSIGN, conf=ABLATION_CONF):
        r['contrast'] = _tag(ch); r['n_contrast'] = len(set(ch))
        rows.append(r)

abl = pd.DataFrame(rows)
abl_csv = os.path.join(OUT_ROOT, f'rvip_ablation_D{ABLATION_DATASET_ID}.csv')
abl.to_csv(abl_csv, index=False)
print('\nwrote', abl_csv)

# ---- rank: median per-point distance (detected) + miss rate per config ----
def _summ(g):
    d = pd.concat([g['d_ant'], g['d_inf']]); det = d.dropna()
    return pd.Series({'median_px': det.median(), 'mean_px': det.mean(),
                      'miss_rate': float(d.isna().mean()), 'n_contrast': g['n_contrast'].iloc[0]})
summary = abl.groupby('contrast').apply(_summ).sort_values('median_px')
summary.to_csv(os.path.join(OUT_ROOT, f'rvip_ablation_D{ABLATION_DATASET_ID}_summary.csv'))
print('\n=== ranked by median point distance (lower = better) ===')
print(summary.round(2))


In [ ]:
import matplotlib.pyplot as plt
# ranked violin of per-point distance (both IPs pooled), best (lowest median) first
order = summary.index.tolist()
long = abl.melt(id_vars=['contrast'], value_vars=['d_ant', 'd_inf'], value_name='dist').dropna(subset=['dist'])
groups = [long[long.contrast == c]['dist'].values for c in order]

plt.figure(figsize=(1.1 * len(order) + 3, 5), dpi=200)
parts = plt.violinplot(groups, showmedians=True, showextrema=False, widths=0.8)
for i, b in enumerate(parts['bodies']):
    b.set_alpha(0.5); b.set_facecolor('C2' if summary.loc[order[i], 'n_contrast'] == 1 else 'C0')
parts['cmedians'].set_color('k')
for i, c in enumerate(order):
    v = groups[i]
    plt.scatter(np.random.normal(i + 1, 0.05, len(v)), v, s=10, alpha=0.5, color='k', zorder=3)
    plt.text(i + 1, np.median(v), f"{summary.loc[c,'median_px']:.1f}", ha='center', va='bottom', fontsize=9)
    plt.text(i + 1, plt.ylim()[1] * 0.97, f"{summary.loc[c,'miss_rate']*100:.0f}% miss", ha='center', va='top', fontsize=8, color='crimson')
plt.xticks(range(1, len(order) + 1), order, rotation=30, ha='right')
plt.ylabel('point distance to GT [px @256]'); plt.xlabel('contrast config (green = single, blue = triple)')
plt.title(f'RVIP contrast ablation — Dataset{ABLATION_DATASET_ID} (ranked)')
plt.grid(axis='y', alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(OUT_ROOT, f'rvip_ablation_D{ABLATION_DATASET_ID}.png'), dpi=200); plt.show()
